# 🛰️ Orbital Anomaly OpenEnv — Training Notebook v4.1

> **Theme 3 — World Modeling** | **Theme 2 — Long-Horizon Planning**

**v4.1 — The correct GRPO fix:**
- Reward function uses a **fixed random seed per prompt index** so all 6 completions
  for the same prompt get identical env dynamics
- No pickle/snapshot corruption
- `model.eval()` only — no Unsloth kernel swaps
- Heuristic priority: eclipse+battery before comms

**You have HF Credits ($30) — use this notebook on HF Spaces or Kaggle (free T4)**


## Cell 1 — Install

In [1]:
%%capture
import subprocess, sys
for cmd in [
    [sys.executable,'-m','pip','install','-q',
     'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'],
    [sys.executable,'-m','pip','install','-q',
     'bitsandbytes','trl>=0.12.0','transformers>=4.45.0',
     'accelerate','peft','datasets','openenv-core','matplotlib','numpy'],
    ['bash','-c',
     'rm -rf /content/repo && git clone -q '
     'https://github.com/umed-indulkar/orbital-anomaly-openenv.git /content/repo'],
]:
    subprocess.run(cmd, check=False)
import sys; sys.path.insert(0, '/content/repo')
print('Done. Restart runtime if first install, then run from Cell 2.')


## Cell 2 — Imports

In [2]:
import os, sys, gc, json, random, logging, warnings, contextlib, io
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
sys.path.insert(0, '/content/repo')

import torch
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import GenerationConfig

with contextlib.redirect_stdout(io.StringIO()):
    from unsloth import FastLanguageModel

from server.orbital_anomaly_openenv_environment import OrbitalAnomalyOpenenvEnvironment
from models import OrbitalAnomalyOpenenvAction

VALID_ACTIONS = ['rotate_to_sun', 'disable_payload', 'reboot_comms',
                 'enter_safe_mode', 'switch_power_bus', 'noop']

print(f'PyTorch  {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Switch to T4 GPU runtime.')
print('Imports OK')


PyTorch  2.10.0+cu128
GPU:     Tesla T4
VRAM:    15.6 GB
Imports OK


## Cell 3 — Environment Check

In [3]:
for task in ['easy', 'medium', 'hard']:
    e = OrbitalAnomalyOpenenvEnvironment()
    o = e.reset(task_id=task)
    s = e.step(OrbitalAnomalyOpenenvAction(action_type='rotate_to_sun'))
    assert 0 < o.reward < 1 and 0 < s.reward < 1
    print(f'  {task:6s}  r={o.reward:.4f}  soc={o.battery_soc:.1f}%  '
          f'temp={o.thermal_temp:.1f}C  sunlit={o.sunlit}')
print('Environment OK ✓')


  easy    r=0.4501  soc=38.0%  temp=38.0C  sunlit=True
  medium  r=0.4501  soc=61.0%  temp=68.0C  sunlit=True
  hard    r=0.4501  soc=22.0%  temp=72.0C  sunlit=False
Environment OK ✓


## Cell 4 — Load Model (Unsloth 4-bit + LoRA)

In [4]:
MODEL_NAME  = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
LORA_RANK   = 16
MAX_SEQ_LEN = 512

print(f'Loading {MODEL_NAME}...')
with contextlib.redirect_stdout(io.StringIO()):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN,
        dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK,
        target_modules=['q_proj','v_proj','k_proj','o_proj',
                         'gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )

# Fix the max_length warning — replace generation_config entirely
model.generation_config = GenerationConfig(
    max_length=None, max_new_tokens=None,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=True,
)
tokenizer.model_max_length = MAX_SEQ_LEN

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({trainable/total:.2%})')
print(f'GPU:       {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Model ready ✓')


Loading unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit...


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Trainable: 18,464,768 / 907,081,216  (2.04%)
GPU:       1.26 GB
Model ready ✓


## Cell 5 — Prompts, Heuristic, Episode Runner
Heuristic priority order: battery+eclipse → thermal → battery_warning → comms → solar


In [5]:
SYSTEM_PROMPT = (
    'You are an autonomous satellite mission control AI.\n'
    'A spacecraft is failing. Choose exactly ONE action.\n\n'
    'Actions:\n'
    '  rotate_to_sun    — realign solar panels (USELESS if sunlit=False)\n'
    '  disable_payload  — cut science payload to reduce heat and power\n'
    '  reboot_comms     — restore RF communications chain\n'
    '  enter_safe_mode  — emergency: disable all non-critical systems\n'
    '  switch_power_bus — activate backup battery (works in eclipse)\n'
    '  noop             — do nothing\n\n'
    'Reply with ONLY the action name. Nothing else.'
)


def make_prompt(obs) -> str:
    meta     = obs.metadata or {}
    beliefs  = meta.get('fault_beliefs', {})
    dominant = meta.get('dominant_subsystem', 'unknown')
    phase    = meta.get('phase', 0)
    top3 = ', '.join(
        f'{k}({v:.0%})' for k, v in
        sorted(beliefs.items(), key=lambda x: x[1], reverse=True)[:3]
    ) if beliefs else 'unknown'
    sunlit = getattr(obs, 'sunlit', True)
    gs     = getattr(obs, 'ground_station_visible', True)
    def lvl(v, lo, hi): return 'CRIT' if v < lo else ('WARN' if v < hi else 'OK')
    return (
        f'Phase {phase+1}/3  Status:{obs.mission_status.upper()}\n'
        f'Battery:{obs.battery_soc:.0f}%[{lvl(obs.battery_soc,20,40)}] '
        f'Solar:{obs.solar_efficiency:.2f}[{lvl(obs.solar_efficiency*100,30,60)}] '
        f'Thermal:{obs.thermal_temp:.0f}C'
        f'[{"CRIT" if obs.thermal_temp>85 else "WARN" if obs.thermal_temp>70 else "OK"}] '
        f'Comms:{obs.comms_signal:.2f}[{lvl(obs.comms_signal*100,30,60)}]\n'
        f'Sunlit:{sunlit}  GS:{gs}  '
        f'Payload:{"ON" if obs.payload_on else "OFF"}  '
        f'Safe:{"ON" if obs.safe_mode else "OFF"}\n'
        f'WorldModel: dominant={dominant}  top3={top3}\nAction:'
    )


def apply_chat(obs) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': make_prompt(obs)}],
        tokenize=False, add_generation_prompt=True
    )


def parse_action(text: str) -> str:
    t = text.strip().lower()
    for a in VALID_ACTIONS:
        if a in t: return a
    return 'noop'


def heuristic(env) -> str:
    """
    Eclipse-aware heuristic. Priority order matters:
    1. Battery + eclipse FIRST (eclipse makes rotate_to_sun useless)
    2. Thermal emergency
    3. Battery warning
    4. Comms (after battery — power > comms)
    5. Solar (only in sunlight)
    """
    bat    = env.battery_soc
    sol    = env.sun_vector_alignment * env.panel_health
    temp   = env.payload_temp
    comms  = max(0.0, min(1.0, 1.0 - env.bit_error_rate * 5.0 - env.packet_loss_ratio))
    sunlit = env.sunlit
    payl   = env.payload_on

    if bat < 12.0:                         return 'switch_power_bus'
    if bat < 30.0 and not sunlit:          return 'switch_power_bus'
    if bat < 20.0 and sol < 0.35:          return 'rotate_to_sun'
    if temp > 84.0:                        return 'enter_safe_mode'
    if temp > 74.0 and payl:               return 'disable_payload'
    if bat < 38.0 and not sunlit:          return 'switch_power_bus'
    if bat < 35.0:                         return 'rotate_to_sun'
    if comms < 0.22:                       return 'reboot_comms'
    if sol < 0.42 and sunlit:              return 'rotate_to_sun'
    if comms < 0.50:                       return 'reboot_comms'
    if temp > 62.0 and payl:               return 'disable_payload'
    if sol < 0.70 and sunlit:              return 'rotate_to_sun'
    return 'noop'


def run_episode(task_id='easy', max_steps=12, temperature=0.7):
    """Evaluation episode. Uses model.eval() only — no Unsloth kernel swap."""
    model.eval()
    env = OrbitalAnomalyOpenenvEnvironment()
    obs = env.reset(task_id=task_id)
    total, rewards, actions = 0.0, [], []

    for _ in range(max_steps):
        if env._check_done(): break
        enc = tokenizer(
            apply_chat(obs), return_tensors='pt',
            truncation=True, max_length=MAX_SEQ_LEN,
            return_attention_mask=True,
        ).to('cuda')
        with torch.no_grad():
            out = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_new_tokens=8, do_sample=True,
                temperature=temperature, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(
            out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        act = parse_action(gen)
        obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
        r   = float(obs.reward or 0.001)
        total += r; rewards.append(r); actions.append(act)

    avg = total / len(rewards) if rewards else 0.001
    return avg, actions, rewards


# Test heuristic on hard task (eclipse, low battery)
env_t = OrbitalAnomalyOpenenvEnvironment()
env_t.reset(task_id='hard')
h = heuristic(env_t)
print(f'Hard task heuristic (eclipse+22% battery): {h}')
assert h == 'switch_power_bus', f'Heuristic wrong: {h}'

# Smoke test
r, acts, _ = run_episode(task_id='easy', max_steps=3)
print(f'Smoke test: avg={r:.4f}  actions={acts}  unique={len(set(acts))}')
print('OK ✓')


Hard task heuristic (eclipse+22% battery): switch_power_bus
Smoke test: avg=0.6855  actions=['reboot_comms', 'disable_payload', 'disable_payload']  unique=2
OK ✓


## Cell 6 — Baseline Evaluation
⚠️ **Do NOT re-run after training (Cell 9).** This captures `pre_d` for comparison.


In [6]:
print('Evaluating baseline...')
print()
baseline, pre_d = {}, {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for _ in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.7)
        rewards.append(r); all_acts.extend(acts)
    baseline[task] = rewards
    pre_d[task] = {a: all_acts.count(a) / max(len(all_acts), 1)
                   for a in VALID_ACTIONS}
    print(f'  {task:6s}  mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique={len(set(all_acts))}/6')

print()
print(f'  easy   BEFORE: {np.mean(baseline["easy"]):.3f}')
print(f'  medium BEFORE: {np.mean(baseline["medium"]):.3f}')
print(f'  hard   BEFORE: {np.mean(baseline["hard"]):.3f}')
print('pre_d captured. DO NOT re-run after training.')


Evaluating baseline...

  easy    mean=0.639  min=0.619  max=0.648  unique=2/6
  medium  mean=0.680  min=0.620  max=0.727  unique=2/6
  hard    mean=0.118  min=0.101  max=0.124  unique=2/6

  easy   BEFORE: 0.639
  medium BEFORE: 0.680
  hard   BEFORE: 0.118
pre_d captured. DO NOT re-run after training.


## Cell 7 — Training Dataset
Each record stores `prompt` + `task_id` + `prompt_seed`.
The reward function uses `prompt_seed` to set numpy/random seeds so all 6 completions
for the same prompt see **the same random env dynamics**.
No pickle corruption. Clean and simple.


In [7]:
def build_dataset(n=300):
    # 60% easy / 20% medium / 20% hard
    # Hard task must appear in training so model sees eclipse + low-battery states
    task_weights = ['easy'] * 6 + ['medium'] * 2 + ['hard'] * 2
    records = []
    base_env = OrbitalAnomalyOpenenvEnvironment()

    for i in range(n):
        task = random.choice(task_weights)
        obs  = base_env.reset(task_id=task)
        # 0-4 random steps for diverse states
        for _ in range(random.randint(0, 4)):
            obs = base_env.step(OrbitalAnomalyOpenenvAction(
                action_type=random.choice(VALID_ACTIONS)))
            if obs.done:
                obs = base_env.reset(task_id=task)
        records.append({
            'prompt':      apply_chat(obs),
            'task_id':     task,
            'prompt_seed': i,
        })

    return Dataset.from_list(records)


print('Building training dataset...')
train_ds = build_dataset(n=300)
task_counts = {}
for r in train_ds:
    task_counts[r['task_id']] = task_counts.get(r['task_id'], 0) + 1
print(f'Dataset: {len(train_ds)} records')
for t, c in sorted(task_counts.items()):
    print(f'  {t}: {c} ({c/len(train_ds):.0%})')
print(f'Sample prompt (200 chars):')
print(train_ds[0]['prompt'][:200])


Building training dataset...
Dataset: 300 records
  easy: 176 (59%)
  hard: 50 (17%)
  medium: 74 (25%)
Sample prompt (200 chars):
<|im_start|>system
You are an autonomous satellite mission control AI.
A spacecraft is failing. Choose exactly ONE action.

Actions:
  rotate_to_sun    — realign solar panels (USELESS if sunlit=False)


## Cell 8 — GRPO Reward Function v4.1

**The correct approach:**
- Use `prompt_seed` to set `random.seed()` before env reset
- All 6 completions for the same prompt use the same seed → same env state
- No pickle, no snapshot, no corruption
- Group advantage = actual action quality difference on that state

Three reward components (anti-hacking per Self-serve guide):
1. Episode reward: 6-step rollout (action at step 0, heuristic after)
2. Eclipse penalty: −0.10 for rotate_to_sun when sunlit=False
3. Format reward: +0.04 for clean single-action output


In [8]:
def satellite_reward(completions, prompts=None, task_id=None,
                     prompt_seed=None, **kwargs):
    """
    GRPO reward v4.2.
    Same deterministic seed approach as v4.1.
    Added: context penalties for physically wrong actions.
      - rotate_to_sun in eclipse: -0.10
      - reboot_comms when comms already dead (< 0.05) AND battery low: -0.12
        This is exactly the hard-task failure mode (comms=0, bat=22%)
      - switch_power_bus in eclipse with low battery: +0.08 bonus
    """
    rewards = []

    task = 'easy'
    if task_id is not None:
        t = task_id[0] if isinstance(task_id, list) else task_id
        if t in ['easy', 'medium', 'hard']: task = t

    seed = 42
    if prompt_seed is not None:
        s = prompt_seed[0] if isinstance(prompt_seed, list) else prompt_seed
        seed = int(s)

    for completion in completions:
        if isinstance(completion, list):
            text = (completion[0].get('content', '') if isinstance(completion[0], dict)
                    else str(completion[0]))
        else:
            text = str(completion)

        action = parse_action(text)

        random.seed(seed)
        np.random.seed(seed)

        env = OrbitalAnomalyOpenenvEnvironment()
        obs = env.reset(task_id=task)

        n_steps = random.randint(0, 4)
        for _ in range(n_steps):
            if obs.done: break
            obs = env.step(OrbitalAnomalyOpenenvAction(
                action_type=random.choice(VALID_ACTIONS)))

        if obs.done:
            random.seed(None)
            rewards.append(0.001)
            continue

        # Capture state for context penalties
        in_eclipse = not env.sunlit
        bat        = env.battery_soc
        comms      = max(0.0, min(1.0, 1.0 - env.bit_error_rate*5.0 - env.packet_loss_ratio))

        # Run episode: step 0 = model action, steps 1-5 = heuristic
        total, steps = 0.0, 0
        for step_i in range(6):
            if env._check_done(): break
            act = action if step_i == 0 else heuristic(env)
            obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
            total += float(obs.reward or 0.001)
            steps += 1

        episode_r = total / max(steps, 1)

        # ── Context penalties (the hard-task fix) ─────────────────────────────
        shape = 0.0

        # 1. rotate_to_sun in eclipse is physically impossible
        if in_eclipse and action == 'rotate_to_sun':
            shape -= 0.10

        # 2. reboot_comms when comms already dead AND battery low = wasted step
        #    Hard task starts: comms=0.00, bat=22%, sunlit=False
        #    Model was doing reboot_comms 69% of the time on hard — this fixes it
        if action == 'reboot_comms' and comms < 0.05 and bat < 35.0:
            shape -= 0.12

        # 3. switch_power_bus in eclipse with low battery = correct action
        if action == 'switch_power_bus' and in_eclipse and bat < 35.0:
            shape += 0.08

        # Format reward
        first_word = text.strip().lower().split()[0] if text.strip() else ''
        fmt_r = 0.04 if first_word in VALID_ACTIONS else 0.0

        rewards.append(episode_r + shape + fmt_r)

    random.seed(None)
    np.random.seed(None)
    return rewards


# Test on hard task — switch_power_bus must beat reboot_comms
print('Testing reward function v4.2...')
test_completions = [[{'content': a}] for a in VALID_ACTIONS]
test_r = satellite_reward(
    test_completions,
    task_id=['hard'] * 6,
    prompt_seed=[0] * 6
)
print('Hard task (seed=0, eclipse+low battery) — action scores:')
for a, r in zip(VALID_ACTIONS, test_r):
    bar = '#' * max(0, int((r - 0.3) * 30))
    print(f'  {a:<22} {r:.4f}  {bar}')
r_switch  = test_r[VALID_ACTIONS.index('switch_power_bus')]
r_reboot  = test_r[VALID_ACTIONS.index('reboot_comms')]
r_rotate  = test_r[VALID_ACTIONS.index('rotate_to_sun')]
print(f'switch_power_bus vs reboot_comms: {r_switch:.4f} vs {r_reboot:.4f}  '
      f'{"GOOD ✓" if r_switch > r_reboot else "STILL WRONG"}')
print(f'switch_power_bus vs rotate_to_sun: {r_switch:.4f} vs {r_rotate:.4f}  '
      f'{"GOOD ✓" if r_switch > r_rotate else "STILL WRONG"}')
spread = max(test_r) - min(test_r)
print(f'Spread: {spread:.4f}  {"GOOD" if spread > 0.05 else "LOW"}')
# Verify determinism
assert satellite_reward(test_completions, task_id=['hard']*6, prompt_seed=[0]*6) == test_r
print('Determinism verified ✓')


Testing reward function v4.2...
Hard task (seed=0, eclipse+low battery) — action scores:
  rotate_to_sun          0.1748  
  disable_payload        0.1646  
  reboot_comms           0.0425  
  enter_safe_mode        0.1646  
  switch_power_bus       0.1660  
  noop                   0.1625  
switch_power_bus vs reboot_comms: 0.1660 vs 0.0425  GOOD ✓
switch_power_bus vs rotate_to_sun: 0.1660 vs 0.1748  STILL WRONG
Spread: 0.1323  GOOD
Determinism verified ✓


## Cell 9 — GRPO Training

In [9]:
from trl import GRPOConfig, GRPOTrainer
import transformers

# Standard PyTorch train mode — no Unsloth kernel swap
model.train()
gc.collect()
torch.cuda.empty_cache()

grpo_cfg = GRPOConfig(
    output_dir                  = '/content/grpo_out',
    num_train_epochs            = 1,
    max_steps                   = 100,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 3e-6,
    num_generations             = 6,
    max_prompt_length           = 450,
    max_completion_length       = 10,
    temperature                 = 1.0,    # high = diverse completions
    beta                        = 0.05,   # low KL = more exploration
    logging_steps               = 5,
    save_steps                  = 50,
    report_to                   = 'none',
    remove_unused_columns       = False,
    dataloader_num_workers      = 0,
    log_completions             = False,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [satellite_reward],
    args             = grpo_cfg,
    train_dataset    = train_ds,
)

print('Starting GRPO training v4.1...')
print(f'  Steps:       {grpo_cfg.max_steps}  (~45-60 min on T4)')
print(f'  Generations: {grpo_cfg.num_generations} per prompt')
print(f'  Key fix:     deterministic seed per prompt = real group advantage')
print(f'  Watch for:   reward_std > 0.05 in logs = learning signal working')
print(f'  GPU before:  {torch.cuda.memory_allocated()/1e9:.2f} GB')
print()

transformers.logging.set_verbosity_error()
train_result = trainer.train()

# Standard eval mode — no kernel swap
model.eval()
gc.collect()
torch.cuda.empty_cache()

print(f'\nTraining complete! Loss: {train_result.training_loss:.6f}')
print(f'GPU after: {torch.cuda.memory_allocated()/1e9:.2f} GB')


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6
Starting GRPO training v4.1...
  Steps:       100  (~45-60 min on T4)
  Generations: 6 per prompt
  Key fix:     deterministic seed per prompt = real group advantage
  Watch for:   reward_std > 0.05 in logs = learning signal working
  GPU before:  1.31 GB

Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '-0.08999', 'grad_norm': '5.448', 'learning_rate': '1.2e-06', 'num_tokens': '2.714e+04', 'completions/mean_length': '3.625', 'completions/min_length': '3', 'completions/max_length': '5', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '3.625', 'completions/min_terminated_length': '3', 'completions/max_terminated_length': '5', 'rewards/satellite_reward/mean': '0.6408', 'rewards/satellite_reward/std': '0.02702', 'reward': '0.6408', 'reward_std': '0.0

## Cell 10 — Training Reward Curve

In [10]:
log_hist = trainer.state.log_history
RKEYS = ['reward', 'rewards', 'mean_reward',
          'rewards/satellite_reward/mean', 'reward/mean']
train_steps, train_rewards, train_std = [], [], []

for entry in log_hist:
    for k in RKEYS:
        if k in entry:
            train_steps.append(entry.get('step', len(train_steps) + 1))
            train_rewards.append(float(entry[k]))
            std_key = 'reward_std' if 'reward_std' in entry else \
                      'rewards/satellite_reward/std'
            train_std.append(float(entry.get(std_key, 0.0)))
            break

if train_rewards:
    print(f'Reward logs: {len(train_rewards)} entries')
    print(f'First: {train_rewards[0]:.4f}  Last: {train_rewards[-1]:.4f}')
    print(f'Trend: {"UP ↑" if train_rewards[-1] > train_rewards[0] else "FLAT"}')
    if train_std:
        avg_std = np.mean(train_std)
        print(f'Avg reward_std: {avg_std:.4f}')
        if avg_std > 0.05:
            print('Good spread in groups — model IS learning per-state actions ✓')
        elif avg_std > 0.02:
            print('Moderate spread — some learning signal')
        else:
            print('Low spread — weak signal (but this is expected with 1.5B model)')
else:
    keys = list(log_hist[0].keys()) if log_hist else []
    print('No reward keys found. Available:', keys)


Reward logs: 20 entries
First: 0.6408  Last: 0.6234
Trend: FLAT
Avg reward_std: 0.0215
Moderate spread — some learning signal


## Cell 11 — Post-Training Evaluation

In [11]:
# model.eval() already set at end of Cell 9
print('Evaluating post-training...')
print()
post = {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for _ in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.8)
        rewards.append(r)
        all_acts.extend(acts)
    post[task] = rewards
    print(f'  {task:6s}  mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique={len(set(all_acts))}/6')

print()
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}  Result')
print('-' * 50)
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    d = a - b
    result = 'IMPROVED' if d > 0.005 else ('similar' if abs(d) < 0.005 else 'WORSE')
    print(f'{task:<10} {b:>8.3f} {a:>8.3f} {d:>+8.3f}  {result}')

print()
print('Hard = zero training prompts. Improvement = generalisation of world model.')


Evaluating post-training...

  easy    mean=0.764  min=0.723  max=0.793  unique=4/6
  medium  mean=0.798  min=0.756  max=0.829  unique=4/6
  hard    mean=0.118  min=0.101  max=0.142  unique=3/6

Task         Before    After    Delta  Result
--------------------------------------------------
easy          0.639    0.764   +0.125  IMPROVED
medium        0.680    0.798   +0.118  IMPROVED
hard          0.118    0.118   +0.000  similar

Hard = zero training prompts. Improvement = generalisation of world model.


## Cell 12 — Training Results Charts

In [12]:
def smooth(v, w=4):
    return [np.mean(v[max(0, i-w):i+1]) for i in range(len(v))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0e1a')
for ax in axes:
    ax.set_facecolor('#111827')
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')
    ax.xaxis.label.set_color('#9ca3af')
    ax.yaxis.label.set_color('#9ca3af')
    ax.title.set_color('#f9fafb')

base_e = np.mean(baseline['easy'])
post_e = np.mean(post['easy'])

if len(train_rewards) >= 3:
    sm = smooth(train_rewards)
    axes[0].plot(train_steps, train_rewards, color='#374151',
                 alpha=0.35, lw=1, label='Raw')
    axes[0].plot(train_steps, sm, color='#3b82f6',
                 lw=2.5, label='Smoothed')
    axes[0].axhline(y=base_e, color='#ef4444', ls='--', lw=1.5,
                    label=f'Eval before: {base_e:.3f}')
    axes[0].axhline(y=post_e, color='#10b981', ls=':', lw=1.5,
                    label=f'Eval after:  {post_e:.3f}')
    axes[0].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
else:
    axes[0].bar(['Before', 'After'], [base_e, post_e],
                color=['#ef4444', '#10b981'], alpha=0.85)
    for xi, val in enumerate([base_e, post_e]):
        axes[0].text(xi, val + 0.008, f'{val:.3f}',
                     ha='center', color='white', fontsize=11)
axes[0].set_title('GRPO Training Curve (Easy Task)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Reward')
axes[0].grid(alpha=0.2, color='#374151')

task_labels = ['Easy', 'Medium', 'Hard']
pre_m  = [np.mean(baseline[t.lower()]) for t in task_labels]
post_m = [np.mean(post[t.lower()])     for t in task_labels]
x, w = np.arange(3), 0.35
axes[1].bar(x - w/2, pre_m,  w, color='#ef4444', alpha=0.85, label='Before')
axes[1].bar(x + w/2, post_m, w, color='#10b981', alpha=0.85, label='After')
for i, (p, q) in enumerate(zip(pre_m, post_m)):
    axes[1].text(i-w/2, p+0.008, f'{p:.3f}', ha='center', color='white', fontsize=9)
    axes[1].text(i+w/2, q+0.008, f'{q:.3f}', ha='center', color='white', fontsize=9)
axes[1].set_title('Pre vs Post — All Tasks', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_labels)
axes[1].set_ylabel('Avg Step Reward')
axes[1].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[1].grid(axis='y', alpha=0.2, color='#374151')

post_d_easy = {a: 0 for a in VALID_ACTIONS}
for _ in range(3):
    _, acts, _ = run_episode(task_id='easy', max_steps=12, temperature=0.8)
    for a in acts:
        post_d_easy[a] = post_d_easy.get(a, 0) + 1
tot = max(sum(post_d_easy.values()), 1)
post_d_easy = {a: post_d_easy[a] / tot for a in VALID_ACTIONS}
short = ['rotate', 'disable', 'reboot', 'safe', 'power', 'noop']
px, w2 = np.arange(6), 0.35
axes[2].bar(px - w2/2, [pre_d['easy'].get(a, 0) for a in VALID_ACTIONS],
            w2, color='#ef4444', alpha=0.85, label='Before')
axes[2].bar(px + w2/2, [post_d_easy.get(a, 0) for a in VALID_ACTIONS],
            w2, color='#10b981', alpha=0.85, label='After')
axes[2].set_title('Action Distribution Shift', fontsize=11, fontweight='bold')
axes[2].set_xticks(px)
axes[2].set_xticklabels(short, fontsize=8, rotation=15)
axes[2].set_ylabel('Fraction of Steps')
axes[2].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[2].grid(axis='y', alpha=0.2, color='#374151')

plt.suptitle('Orbital Anomaly OpenEnv — Training Results (GRPO v4.1)',
             fontsize=13, color='#f9fafb', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved training_results.png')
try:
    from google.colab import files
    files.download('training_results.png')
except Exception:
    pass


Saved training_results.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 13 — Behavioral Analysis

In [13]:
print('=== BEHAVIORAL ANALYSIS ===')
print()
post_d = {}
for task in ['easy', 'medium', 'hard']:
    all_acts = []
    for _ in range(3):
        _, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.8)
        all_acts.extend(acts)
    post_d[task] = {a: all_acts.count(a) / max(len(all_acts), 1)
                    for a in VALID_ACTIONS}

if pre_d and 'easy' in pre_d:
    for task in ['easy', 'medium']:
        print(f'{task.upper()} — before vs after:')
        print(f'  {"Action":<22} {"Before":>8} {"After":>8}  Change')
        print('  ' + '-' * 46)
        for act in VALID_ACTIONS:
            b = pre_d[task].get(act, 0)
            a = post_d[task].get(act, 0)
            d = a - b
            arrow = 'UP' if d > 0.05 else ('DOWN' if d < -0.05 else '~')
            print(f'  {act:<22} {b:>8.1%} {a:>8.1%}  {arrow}  ({d:+.1%})')
        print()

print('HARD — post-training distribution:')
for act in VALID_ACTIONS:
    freq = post_d['hard'].get(act, 0)
    if freq > 0:
        bar = '#' * int(freq * 20)
        print(f'  {act:<22} {bar:<20} {freq:.0%}')

print()
n_easy = sum(1 for v in post_d['easy'].values() if v > 0)
n_hard = sum(1 for v in post_d['hard'].values() if v > 0)
print(f'Action diversity: easy={n_easy}/6  hard={n_hard}/6')
best_easy = max(post_d['easy'], key=post_d['easy'].get)
best_hard = max(post_d['hard'], key=post_d['hard'].get)
print(f'Easy top: [{best_easy}]  Hard top: [{best_hard}]')
if best_hard != best_easy:
    print('Context-aware behaviour: different top actions per task = world modeling ✓')
print()
print('KEY METRICS:')
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    print(f'  {task}: {b:.3f} → {a:.3f}  ({(a-b)/b*100:+.1f}%)')


=== BEHAVIORAL ANALYSIS ===

EASY — before vs after:
  Action                   Before    After  Change
  ----------------------------------------------
  rotate_to_sun              0.0%    55.6%  UP  (+55.6%)
  disable_payload           82.5%    30.6%  DOWN  (-51.9%)
  reboot_comms              17.5%    13.9%  ~  (-3.6%)
  enter_safe_mode            0.0%     0.0%  ~  (+0.0%)
  switch_power_bus           0.0%     0.0%  ~  (+0.0%)
  noop                       0.0%     0.0%  ~  (+0.0%)

MEDIUM — before vs after:
  Action                   Before    After  Change
  ----------------------------------------------
  rotate_to_sun              0.0%    58.3%  UP  (+58.3%)
  disable_payload           86.7%    25.0%  DOWN  (-61.7%)
  reboot_comms              13.3%    13.9%  ~  (+0.6%)
  enter_safe_mode            0.0%     0.0%  ~  (+0.0%)
  switch_power_bus           0.0%     2.8%  ~  (+2.8%)
  noop                       0.0%     0.0%  ~  (+0.0%)

HARD — post-training distribution:
  rotate_to_

## Cell 14 — Fault Belief State (Theme 3: World Modeling)

In [14]:
print('FAULT BELIEF STATE — hard task, 4 steps')
print()
env = OrbitalAnomalyOpenenvEnvironment()
obs = env.reset(task_id='hard')

for sn in range(4):
    beliefs  = obs.metadata.get('fault_beliefs', {})
    dominant = obs.metadata.get('dominant_subsystem', 'unknown')
    print(f'Step {sn}  BAT={obs.battery_soc:.0f}%  TEMP={obs.thermal_temp:.0f}C  '
          f'COMMS={obs.comms_signal:.2f}  SUNLIT={obs.sunlit}')
    print(f'  Dominant: {dominant}')
    for fault, prob in sorted(beliefs.items(), key=lambda x: x[1], reverse=True):
        bar = chr(9608)*int(prob*18) + chr(9617)*(18-int(prob*18))
        print(f'  {fault:30s} {bar} {prob*100:4.0f}%')
    print()
    if sn < 3:
        act = heuristic(env)
        obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
        print(f'  → heuristic chose: {act}\n')


FAULT BELIEF STATE — hard task, 4 steps

Step 0  BAT=22%  TEMP=72C  COMMS=0.00  SUNLIT=False
  Dominant: Comms
  mppt_stuck                     ██████████████████  100%
  transponder_overheating        █████████████████░  100%
  reaction_wheel_saturation      ███████████████░░░   88%
  radiator_valve_stuck           ██████████████░░░░   82%
  panel_deployment_jam           ████████████░░░░░░   71%
  antenna_gimbal_stall           ████████████░░░░░░   68%
  amplifier_degradation          ███████████░░░░░░░   65%
  bus_short_transient            ██████████░░░░░░░░   60%
  heat_pipe_failure              ██████████░░░░░░░░   57%
  heater_relay_latch             ████████░░░░░░░░░░   48%
  battery_aging                  ███████░░░░░░░░░░░   39%
  gyro_drift                     ███████░░░░░░░░░░░   39%
  star_tracker_dropout           ██████░░░░░░░░░░░░   36%

  → heuristic chose: switch_power_bus

Step 1  BAT=20%  TEMP=75C  COMMS=0.00  SUNLIT=True
  Dominant: Comms
  mppt_stuck              

## Cell 15 — Extended Mission Mode (Theme 2: Long-Horizon)

In [15]:
print('EXTENDED MISSION MODE — 36 steps, 3 phases')
print('Battery + thermal carry over between phases')
print()
env = OrbitalAnomalyOpenenvEnvironment()
obs = env.reset(task_id='easy')
prev_phase = 0
phase_rewards = {0: [], 1: [], 2: []}

for sn in range(36):
    act = heuristic(env)
    obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
    phase = obs.metadata.get('phase', 0)
    phase_rewards[min(phase, 2)].append(obs.reward)
    if phase != prev_phase:
        print(f'>>> PHASE {phase} at step {sn+1}')
        print(f'    SOC:  {obs.battery_soc:.1f}%')
        print(f'    Temp: {obs.thermal_temp:.1f}C')
        prev_phase = phase
    if obs.done:
        print(f'Done at step {sn+1}')
        break

print()
names = ['EPS Crisis', 'Thermal Crisis', 'Comms Crisis']
for ph, rr in phase_rewards.items():
    if rr:
        print(f'  Phase {ph} ({names[ph]:<15}): '
              f'avg={sum(rr)/len(rr):.4f} ({len(rr)} steps)')
print()
print('Poor Phase 0 → lower SOC in Phase 1 → harder thermal management')
print('Cannot optimise phases independently — genuine long-horizon reasoning.')


EXTENDED MISSION MODE — 36 steps, 3 phases
Battery + thermal carry over between phases

>>> PHASE 1 at step 12
    SOC:  38.9%
    Temp: 59.1C
>>> PHASE 2 at step 24
    SOC:  42.9%
    Temp: 52.1C
Done at step 36

  Phase 0 (EPS Crisis     ): avg=0.7262 (11 steps)
  Phase 1 (Thermal Crisis ): avg=0.6555 (12 steps)
  Phase 2 (Comms Crisis   ): avg=0.5626 (13 steps)

Poor Phase 0 → lower SOC in Phase 1 → harder thermal management
Cannot optimise phases independently — genuine long-horizon reasoning.


## Cell 16 — Save LoRA Adapters

In [16]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/orbital_lora_v41'
except Exception:
    SAVE_PATH = '/content/orbital_lora_v41'

os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

summary = {
    'version': '4.1',
    'model':   MODEL_NAME,
    'lora_rank': LORA_RANK,
    'grpo_steps': grpo_cfg.max_steps,
    'baseline': {t: float(np.mean(v)) for t, v in baseline.items()},
    'post':     {t: float(np.mean(v)) for t, v in post.items()},
    'improvement': {
        t: f'{(np.mean(post[t])-np.mean(baseline[t]))/np.mean(baseline[t])*100:+.1f}%'
        for t in ['easy', 'medium', 'hard']
    },
    'train_rewards': train_rewards,
}
with open(f'{SAVE_PATH}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved → {SAVE_PATH}')
print('Improvements:')
for t, v in summary['improvement'].items():
    print(f'  {t}: {v}')


Saved → /content/orbital_lora_v41
Improvements:
  easy: +19.6%
  medium: +17.3%
  hard: +0.4%


## Cell 17 — Final Summary

In [17]:
print('=' * 60)
print('ORBITAL ANOMALY OPENENV — v4.1 TRAINING COMPLETE')
print('=' * 60)
print()
print('RESULTS:')
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    p = (a - b) / b * 100
    sym = 'IMPROVED ✓' if p > 0.5 else ('FLAT' if abs(p) < 0.5 else 'WORSE')
    print(f'  {task:<8}: {b:.3f} → {a:.3f}  ({p:+.1f}%)  {sym}')

print()
print('v4.1 FIXES:')
print('  Reward fn uses prompt_seed → deterministic state → real group advantage')
print('  Heuristic: eclipse+battery BEFORE comms in priority order')
print('  model.eval() only — no Unsloth kernel swapping')
print()
print('THEMES:')
print('  Theme 3 (World Modeling):  13-fault belief state per step in metadata')
print('  Theme 2 (Long-Horizon):    36-step 3-phase Extended Mission Mode')
print('  Theme 1 (Multi-Agent):     Commander + EPS/Thermal/Comms specialists')
print()
print('NEXT: Write HuggingFace blog post with training_results.png')


ORBITAL ANOMALY OPENENV — v4.1 TRAINING COMPLETE

RESULTS:
  easy    : 0.639 → 0.764  (+19.6%)  IMPROVED ✓
  medium  : 0.680 → 0.798  (+17.3%)  IMPROVED ✓
  hard    : 0.118 → 0.118  (+0.4%)  FLAT

v4.1 FIXES:
  Reward fn uses prompt_seed → deterministic state → real group advantage
  Heuristic: eclipse+battery BEFORE comms in priority order
  model.eval() only — no Unsloth kernel swapping

THEMES:
  Theme 3 (World Modeling):  13-fault belief state per step in metadata
  Theme 2 (Long-Horizon):    36-step 3-phase Extended Mission Mode
  Theme 1 (Multi-Agent):     Commander + EPS/Thermal/Comms specialists

NEXT: Write HuggingFace blog post with training_results.png
